# 📚 SEG-Y Volume Processing Notebook

This notebook provides a clean, modular, and reusable pipeline to process SEG-Y seismic data for preprocessing tasks like inspection, organization, visualization, and patch extraction. It follows **functional programming principles**, so each operation is encapsulated in a standalone function.

## 🧭 Notebook Outline

1. [Setup & Configuration](#setup)
2. [Read SEG-Y File](#read)
3. [Parse Headers & Metadata](#headers)
4. [Build 3D Seismic Volume](#volume)
5. [Visualize Inline Slices](#visualize)
6. [Extract 3D Patches to HDF5 (Multiprocessing)](#patches)


## 🔧 Setup & Configuration <a name="setup"></a>

Import required libraries and define constants.


In [ ]:
import numpy as np
import h5py
from obspy.io.segy.segy import _read_segy
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import matplotlib.colors as mcolors

import os
from PIL import Image
import plotly.graph_objects as go

from patch_utils import init_worker, extract_patch
sample_interval_ms = 4  # 4000 µs


## 📥 Read SEG-Y File <a name="read"></a>

Function to load SEG-Y using ObsPy and return a trace stream.


In [ ]:
def read_segy_file(path):
    """Read SEG-Y file using ObsPy."""
    return _read_segy(path, headonly=False)


In [ ]:
# 🔍 Load SEG-Y file
stream = read_segy_file("/Volumes/SSD/data/F3.segy")


## 🗂 Parse Headers & Metadata <a name="headers"></a>

Extract sorted inline and crossline numbers, and create lookup mappings.


In [ ]:
def parse_headers(stream):
    """Extract inline/xline info and mappings."""
    inlines = set()
    xlines = set()
    trace_map = {}
    
    for tr in stream.traces:
        iline = tr.header.for_3d_poststack_data_this_field_is_for_in_line_number
        xline = tr.header.for_3d_poststack_data_this_field_is_for_cross_line_number
        inlines.add(iline)
        xlines.add(xline)
        trace_map[(iline, xline)] = tr.data

    sorted_inlines = sorted(inlines)
    sorted_xlines = sorted(xlines)
    iline_to_idx = {v: i for i, v in enumerate(sorted_inlines)}
    xline_to_idx = {v: i for i, v in enumerate(sorted_xlines)}
    
    return trace_map, sorted_inlines, sorted_xlines, iline_to_idx, xline_to_idx


In [ ]:
# 🧠 Parse headers
trace_map, sorted_inlines, sorted_xlines, iline_to_idx, xline_to_idx = parse_headers(stream)

In [ ]:
trace_map, sorted_inlines, sorted_xlines, iline_to_idx, xline_to_idx = parse_headers(stream)

In [ ]:
masks_path = "/Volumes/SSD/data/F3_BIG/masks"

## 🧱 Build 3D Seismic Volume <a name="volume"></a>

Reorganize SEG-Y trace data into a structured 3D NumPy volume: `[inline, crossline, samples]`.


In [ ]:
def build_volume(trace_map, iline_to_idx, xline_to_idx, n_samples):
    """Construct 3D NumPy volume from trace map."""
    volume = np.full((len(iline_to_idx), len(xline_to_idx), n_samples), np.nan, dtype=np.float32)
    for (iline, xline), data in trace_map.items():
        i = iline_to_idx[iline]
        j = xline_to_idx[xline]
        volume[i, j, :] = data
    return volume


In [ ]:
# 🧱 Build volume
n_samples = next(iter(trace_map.values())).shape[0]
volume = build_volume(trace_map, iline_to_idx, xline_to_idx, n_samples)

In [ ]:
def build_mask_volume(masks_path, sorted_inlines, sorted_xlines, n_samples):
    """Build a 3D mask volume from mask images."""
    mask_volume = np.zeros((len(sorted_inlines), len(sorted_xlines), n_samples), dtype=np.uint8)
    
    for iline in sorted_inlines:
        mask_path = os.path.join(masks_path, f"inline_{iline}_mask.png")
        if os.path.exists(mask_path):
            mask_image = Image.open(mask_path).convert("L")  # Convert to grayscale
            mask_array = np.array(mask_image, dtype=np.uint8)
            
            if mask_array.shape[1] != len(sorted_xlines) or mask_array.shape[0] != n_samples:
                raise ValueError(f"Mask dimensions for inline {iline} do not match the seismic volume.")
            
            iline_idx = iline_to_idx[iline]
            mask_volume[iline_idx, :, :] = mask_array.T  # Transpose to match volume orientation
        else:
            raise FileNotFoundError(f"Mask file for inline {iline} not found at {mask_path}.")
    
    return mask_volume

# Build the mask volume
mask_volume = build_mask_volume(masks_path, sorted_inlines, sorted_xlines, n_samples)

In [ ]:
mask_volume.shape, volume.shape

## 🖼 Visualize Inline Slice (Interactive) <a name="visualize"></a>

Interactive viewer using ipywidgets to scroll through inline sections.


In [ ]:
def create_inline_viewer(volume, sorted_inlines, sorted_xlines, mask_volume=None, mask_cmap=None):
    """Create an interactive inline viewer with optional mask overlay."""
    
    iline_to_idx = {iline: i for i, iline in enumerate(sorted_inlines)}
    xline_range = sorted_xlines
    if mask_volume is not None and mask_cmap is None:
            unique_labels = np.unique(mask_volume)
            mask_cmap = mcolors.ListedColormap(np.random.rand(len(unique_labels), 3))
    def plot_inline(iline_number):
        iline_index = iline_to_idx[iline_number]
        image = volume[iline_index, :, :].T

        plt.figure(figsize=(12, 6))
        plt.imshow(
            image,
            cmap='gray',
            aspect='auto',
            extent=[min(xline_range), max(xline_range), sample_interval_ms * (image.shape[0] - 1), 0],
            vmin=np.nanmin(volume),
            vmax=np.nanmax(volume)
        )
        plt.title(f"Inline {iline_number}")
        plt.xlabel("Crossline")
        plt.ylabel("Time (ms)")
        plt.colorbar(label="Amplitude")

        if mask_volume is not None:
            mask = mask_volume[iline_index, :, :].T
            plt.imshow(
                mask,
                cmap=mask_cmap,
                alpha=0.5,
                aspect='auto',
                extent=[min(xline_range), max(xline_range), sample_interval_ms * (image.shape[0] - 1), 0]
            )

        plt.show()

    slider = IntSlider(min=min(sorted_inlines), max=max(sorted_inlines),
                       step=1, value=sorted_inlines[len(sorted_inlines) // 2], description="Inline")
    interact(plot_inline, iline_number=slider)



In [ ]:
unique_labels = np.unique(mask_volume)
mask_cmap = mcolors.ListedColormap(np.random.rand(len(unique_labels), 3))


In [ ]:

# Define colors: black for 0, and distinguishable colors for other labels
colors = ['black', 'red', 'blue', 'green', 'yellow', 'purple', 'orange', 'cyan', 'magenta', 'brown']
mask_cmap = mcolors.ListedColormap(colors[:len(unique_labels)])
mask_cmap

In [ ]:
x = [(0,10),(0,10),(0,10)]
volume[*x]

In [ ]:
# 🖼 Explore inline viewer

create_inline_viewer(volume, sorted_inlines, sorted_xlines, mask_volume=mask_volume, mask_cmap=mask_cmap)

## 📦 Index Patching<a name="index_patches"></a>
Creating 3D patches using only the index

In [ ]:
def generate_patch_centers(volume_shape, patch_size, stride):
    """
    Compute valid patch center indices for a 3D volume using vectorized NumPy ops.

    Parameters:
    - volume_shape (tuple of int): (inlines, xlines, samples)
    - patch_size (tuple of int): Size of the patch in each dimension
    - stride (tuple of int): Stride between patch centers

    Returns:
    - centers (np.ndarray): Array of shape (N, 3) containing all valid patch centers
    """
    assert len(volume_shape) == 3
    assert len(patch_size) == 3
    assert len(stride) == 3

    starts = [np.arange(patch_size[d] // 2,
                        volume_shape[d] - (patch_size[d] - 1) // 2,
                        stride[d])
              for d in range(3)]

    ii, jj, kk = np.meshgrid(starts[0], starts[1], starts[2], indexing='ij')
    centers = np.stack([ii.ravel(), jj.ravel(), kk.ravel()], axis=1)
    return centers

In [ ]:
def extract_patches_batch(volume, centers, patch_size):
    """
    Extract multiple 3D patches from a volume.

    Parameters:
    - volume (np.ndarray): 3D array (inlines, xlines, samples)
    - centers (np.ndarray): array of shape (N, 3), each row is (i, j, k)
    - patch_size (tuple): (d_inline, d_xline, d_sample)

    Returns:
    - patches (np.ndarray): array of shape (N, d_inline, d_xline, d_sample)
    """
    assert volume.ndim == 3
    assert centers.ndim == 2 and centers.shape[1] == 3
    assert len(patch_size) == 3

    n_patches = centers.shape[0]
    patches = np.empty((n_patches, *patch_size), dtype=volume.dtype)
    half_size = [s // 2 for s in patch_size]

    for idx, (i, j, k) in enumerate(centers):
        start = (i - half_size[0], j - half_size[1], k - half_size[2])
        end = (start[0] + patch_size[0], start[1] + patch_size[1], start[2] + patch_size[2])

        # Bounds check (optional: you can pad instead of error)
        for dim in range(3):
            if start[dim] < 0 or end[dim] > volume.shape[dim]:
                raise ValueError(f"Patch {idx} is out of bounds in dimension {dim}")

        patches[idx] = volume[start[0]:end[0], start[1]:end[1], start[2]:end[2]]

    return patches

In [ ]:
volume_shape = volume.shape
patch_size = (64, 64, 64)
stride = (8,8,8)

centers = generate_patch_centers(volume_shape, patch_size, stride)
print(f"Total patches: {len(centers)}")
print("First 5 centers:", centers[:5])
patches = extract_patches_batch(volume, centers[:5], patch_size)
print("Patches size:",patches.shape)